# Member D: LightGBM, MLP, and Bayesian Optimization

**Owner:** Member D  
**Models:** LightGBM (boosting), MLP (neural network)  
**Task:** Model training, comparison, and Bayesian hyperparameter optimization

## Overview

This notebook covers:
1. **LightGBM Model** - Gradient boosting with early stopping
2. **MLP Model** - Neural network with loss curve tracking
3. **Model Comparison** - Training and evaluation against shared harness
4. **Bayesian Optimization** - Optuna hyperparameter search on winner
5. **Artifacts Export** - Final model and preprocessing pipeline

## Google Colab Setup

This notebook is designed to run on **Google Colab** for GPU acceleration.

### Instructions:
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → GPU
3. Upload project data to Google Drive or use directly
4. Run all cells in sequence

## 1. Setup and Imports

In [ ]:
# Google Colab setup
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set project path (adjust as needed)
    PROJECT_PATH = '/content/drive/MyDrive/ml_hackathon'
    
except ImportError:
    IN_COLAB = False
    print("Running locally")
    PROJECT_PATH = '/home/bharath/Desktop/projects/ml_hackathon'

In [ ]:
# Install required packages (Colab)
if IN_COLAB:
    !pip install -q lightgbm optuna pyyaml pyarrow joblib
    
# Standard imports
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

# Add project to path
sys.path.insert(0, PROJECT_PATH)

# Data handling
import numpy as np
import pandas as pd
import yaml

# ML libraries
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, log_loss, accuracy_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
import joblib

# Hyperparameter optimization
import optuna
from optuna.samplers import TPESampler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print(f"Project path: {PROJECT_PATH}")
print(f"Working directory: {os.getcwd()}")

## 2. Configuration and Data Loading

In [ ]:
# Load configuration files
def load_config(filepath: str) -> Dict[str, Any]:
    """Load YAML configuration file."""
    with open(filepath, 'r') as f:
        return yaml.safe_load(f)

# Load configurations
configs_dir = Path(PROJECT_PATH) / 'configs'

# Load partitions (training/student split)
partitions_config = load_config(configs_dir / 'partitions.yaml')
print("Partitions config loaded:")
print(f"  - Train trips: {len(partitions_config.get('train_trips', []))} trips")
print(f"  - Student test trips: {len(partitions_config.get('student_trips', []))} trips")

# Load model configs
lgbm_config = load_config(configs_dir / 'models' / 'lgbm.yaml') or {}
mlp_config = load_config(configs_dir / 'models' / 'mlp.yaml') or {}
optuna_config = load_config(configs_dir / 'optuna.yaml') or {}

print("\nModel configs loaded")
print(f"  - LightGBM config: {lgbm_config}")
print(f"  - MLP config: {mlp_config}")
print(f"  - Optuna config: {optuna_config}")

In [ ]:
# Load features data
features_path = Path(PROJECT_PATH) / 'data' / 'features' / 'features.parquet'

if features_path.exists():
    print(f"Loading features from: {features_path}")
    features_df = pd.read_parquet(features_path)
    print(f"Features shape: {features_df.shape}")
    print(f"Columns: {features_df.columns.tolist()}")
    print(f"\nPartition distribution:")
    print(features_df['partition'].value_counts())
    print(f"\nTrip distribution:")
    print(features_df.groupby('partition')['trip_id'].nunique())
else:
    print(f"Features file not found at {features_path}")
    print("Please run Member B's feature pipeline first (pipelines/03_build_features.py)")
    print("\nUsing placeholder data for demonstration...")
    
    # Create placeholder data for testing
    np.random.seed(42)
    n_samples = 10000
    n_features = 10
    
    features_df = pd.DataFrame({
        'trip_id': np.random.randint(0, 80, n_samples),
        'partition': np.random.choice(['train', 'student'], n_samples, p=[0.8, 0.2]),
        'point_idx': np.random.randint(0, 100, n_samples),
        'candidate_arc_id': np.random.randint(0, 100, n_samples),
        'label': np.random.randint(0, 2, n_samples),
        **{f'feature_{i}': np.random.randn(n_samples) for i in range(n_features)}
    })
    
    print(f"\nPlaceholder features shape: {features_df.shape}")

In [ ]:
# Prepare feature columns
# Identify feature columns (excluding metadata)
metadata_cols = ['trip_id', 'partition', 'point_idx', 'candidate_arc_id', 'label']
feature_cols = [col for col in features_df.columns if col not in metadata_cols]

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# Check for missing values
missing_pct = features_df[feature_cols].isnull().sum() / len(features_df) * 100
print(f"\nMissing values (%):")
print(missing_pct[missing_pct > 0].to_string() if missing_pct.sum() > 0 else "No missing values")

## 3. Data Splitting (GroupKFold)

In [ ]:
# Split data by partition
train_df = features_df[features_df['partition'] == 'train'].copy()
student_df = features_df[features_df['partition'] == 'student'].copy()

print(f"Training set: {len(train_df)} samples from {train_df['trip_id'].nunique()} trips")
print(f"Student test set: {len(student_df)} samples from {student_df['trip_id'].nunique()} trips")

# Prepare X, y, and groups for training
X_train = train_df[feature_cols].values
y_train = train_df['label'].values
trip_ids_train = train_df['trip_id'].values

X_student = student_df[feature_cols].values
y_student = student_df['label'].values
trip_ids_student = student_df['trip_id'].values

print(f"\nTraining set shape: X={X_train.shape}, y={y_train.shape}")
print(f"Student test set shape: X={X_student.shape}, y={y_student.shape}")
print(f"\nLabel distribution (train): {np.bincount(y_train)}")
print(f"Label distribution (student): {np.bincount(y_student)}")

In [ ]:
# Create GroupKFold splits
n_splits = 5
seed = partitions_config.get('seed', 42)

gkf = GroupKFold(n_splits=n_splits)

# Generate fold indices
folds = list(gkf.split(X_train, y_train, groups=trip_ids_train))

print(f"GroupKFold with {n_splits} folds")
print(f"Random seed: {seed}")
print("\nFold statistics:")

for i, (train_idx, val_idx) in enumerate(folds):
    train_trips = np.unique(trip_ids_train[train_idx])
    val_trips = np.unique(trip_ids_train[val_idx])
    print(f"  Fold {i+1}: {len(train_idx)} train ({len(train_trips)} trips), {len(val_idx)} val ({len(val_trips)} trips)")

## 4. LightGBM Model

### 4.1 Default Hyperparameters

In [ ]:
# LightGBM default parameters
lgbm_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'max_depth': -1,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.0,
    'reg_lambda': 0.0,
    'random_state': seed,
    'n_jobs': -1,
    'verbose': -1,
}

print("LightGBM Parameters:")
for key, value in lgbm_params.items():
    print(f"  {key}: {value}")

### 4.2 Cross-Validation Training

In [ ]:
# Train LightGBM with cross-validation
lgbm_cv_results = []
lgbm_models = []

print("Training LightGBM with 5-fold CV...")
print("=" * 60)

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\nFold {fold_idx + 1}/{n_splits}")
    print("-" * 40)
    
    # Split data
    X_fold_train = X_train[train_idx]
    y_fold_train = y_train[train_idx]
    X_fold_val = X_train[val_idx]
    y_fold_val = y_train[val_idx]
    
    # Create LightGBM datasets
    train_data = lgb.Dataset(X_fold_train, label=y_fold_train)
    val_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_data)
    
    # Train model
    start_time = time.time()
    
    model = lgb.train(
        params={k: v for k, v in lgbm_params.items() if k not in ['n_estimators', 'verbose']},
        train_set=train_data,
        num_boost_round=lgbm_params['n_estimators'],
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=10, verbose=False),
            lgb.log_evaluation(period=0)  # Silent
        ]
    )
    
    train_time = time.time() - start_time
    
    # Predictions
    start_time = time.time()
    y_val_pred_proba = model.predict(X_fold_val, num_iteration=model.best_iteration)
    infer_time = time.time() - start_time
    
    y_val_pred = (y_val_pred_proba >= 0.5).astype(int)
    
    # Metrics
    fold_metrics = {
        'fold': fold_idx + 1,
        'macro_f1': f1_score(y_fold_val, y_val_pred, average='macro'),
        'log_loss': log_loss(y_fold_val, y_val_pred_proba),
        'accuracy': accuracy_score(y_fold_val, y_val_pred),
        'precision': precision_score(y_fold_val, y_val_pred),
        'recall': recall_score(y_fold_val, y_val_pred),
        'best_iteration': model.best_iteration,
        'train_time_s': train_time,
        'infer_time_s': infer_time,
    }
    
    lgbm_cv_results.append(fold_metrics)
    lgbm_models.append(model)
    
    print(f"  Macro F1: {fold_metrics['macro_f1']:.4f}")
    print(f"  Log Loss: {fold_metrics['log_loss']:.4f}")
    print(f"  Best iteration: {model.best_iteration}")
    print(f"  Train time: {train_time:.2f}s")

# Aggregate CV results
lgbm_cv_df = pd.DataFrame(lgbm_cv_results)
print("\n" + "=" * 60)
print("LightGBM CV Results Summary:")
print("=" * 60)
print(f"Macro F1: {lgbm_cv_df['macro_f1'].mean():.4f} (+/- {lgbm_cv_df['macro_f1'].std():.4f})")
print(f"Log Loss: {lgbm_cv_df['log_loss'].mean():.4f} (+/- {lgbm_cv_df['log_loss'].std():.4f})")
print(f"Accuracy: {lgbm_cv_df['accuracy'].mean():.4f}")
print(f"Total train time: {lgbm_cv_df['train_time_s'].sum():.2f}s")

### 4.3 Feature Importance

In [ ]:
# Feature importance from best fold
best_fold_idx = lgbm_cv_df['macro_f1'].idxmax()
best_lgbm_model = lgbm_models[best_fold_idx]

# Get feature importance
lgbm_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_lgbm_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(10, 8))
sns.barplot(data=lgbm_importance.head(20), x='importance', y='feature')
plt.title('LightGBM Feature Importance (Top 20)')
plt.xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

print("\nTop 10 Important Features:")
print(lgbm_importance.head(10))

## 5. MLP Model

### 5.1 Default Hyperparameters

In [ ]:
# MLP default parameters
mlp_params = {
    'hidden_layer_sizes': (64, 32),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 256,
    'learning_rate': 'constant',
    'learning_rate_init': 0.001,
    'max_iter': 100,
    'tol': 1e-4,
    'random_state': seed,
}

print("MLP Parameters:")
for key, value in mlp_params.items():
    print(f"  {key}: {value}")

### 5.2 Cross-Validation Training with Loss Tracking

In [ ]:
# Train MLP with cross-validation and loss tracking
mlp_cv_results = []
mlp_models = []
mlp_scalers = []
mlp_loss_histories = []

print("Training MLP with 5-fold CV...")
print("=" * 60)

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\nFold {fold_idx + 1}/{n_splits}")
    print("-" * 40)
    
    # Split data
    X_fold_train = X_train[train_idx]
    y_fold_train = y_train[train_idx]
    X_fold_val = X_train[val_idx]
    y_fold_val = y_train[val_idx]
    
    # Scale features (fit on training only)
    scaler = StandardScaler()
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)
    
    # Initialize MLP
    mlp = MLPClassifier(
        hidden_layer_sizes=mlp_params['hidden_layer_sizes'],
        activation=mlp_params['activation'],
        solver=mlp_params['solver'],
        alpha=mlp_params['alpha'],
        batch_size=mlp_params['batch_size'],
        learning_rate=mlp_params['learning_rate'],
        learning_rate_init=mlp_params['learning_rate_init'],
        max_iter=1,  # Manual epoch control
        warm_start=True,
        random_state=mlp_params['random_state'],
    )
    
    # Manual epoch training with loss tracking
    train_losses = []
    val_losses = []
    
    start_time = time.time()
    
    for epoch in range(mlp_params['max_iter']):
        # Train one epoch
        mlp.fit(X_fold_train_scaled, y_fold_train)
        
        # Calculate losses
        train_proba = mlp.predict_proba(X_fold_train_scaled)
        val_proba = mlp.predict_proba(X_fold_val_scaled)
        
        train_loss = log_loss(y_fold_train, train_proba)
        val_loss = log_loss(y_fold_val, val_proba)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        # Early stopping check (simplified)
        if epoch > 10 and val_loss > min(val_losses[-10:]):
            break
    
    train_time = time.time() - start_time
    
    # Predictions
    start_time = time.time()
    y_val_pred_proba = mlp.predict_proba(X_fold_val_scaled)[:, 1]
    infer_time = time.time() - start_time
    
    y_val_pred = (y_val_pred_proba >= 0.5).astype(int)
    
    # Metrics
    fold_metrics = {
        'fold': fold_idx + 1,
        'macro_f1': f1_score(y_fold_val, y_val_pred, average='macro'),
        'log_loss': log_loss(y_fold_val, y_val_pred_proba),
        'accuracy': accuracy_score(y_fold_val, y_val_pred),
        'precision': precision_score(y_fold_val, y_val_pred),
        'recall': recall_score(y_fold_val, y_val_pred),
        'n_epochs': len(train_losses),
        'train_time_s': train_time,
        'infer_time_s': infer_time,
    }
    
    mlp_cv_results.append(fold_metrics)
    mlp_models.append(mlp)
    mlp_scalers.append(scaler)
    mlp_loss_histories.append({
        'train': train_losses,
        'val': val_losses
    })
    
    print(f"  Macro F1: {fold_metrics['macro_f1']:.4f}")
    print(f"  Log Loss: {fold_metrics['log_loss']:.4f}")
    print(f"  Epochs trained: {len(train_losses)}")
    print(f"  Train time: {train_time:.2f}s")

# Aggregate CV results
mlp_cv_df = pd.DataFrame(mlp_cv_results)
print("\n" + "=" * 60)
print("MLP CV Results Summary:")
print("=" * 60)
print(f"Macro F1: {mlp_cv_df['macro_f1'].mean():.4f} (+/- {mlp_cv_df['macro_f1'].std():.4f})")
print(f"Log Loss: {mlp_cv_df['log_loss'].mean():.4f} (+/- {mlp_cv_df['log_loss'].std():.4f})")
print(f"Accuracy: {mlp_cv_df['accuracy'].mean():.4f}")
print(f"Total train time: {mlp_cv_df['train_time_s'].sum():.2f}s")

### 5.3 Training/Validation Loss Curves

In [ ]:
# Plot loss curves for best fold
best_fold_idx = mlp_cv_df['macro_f1'].idxmax()
best_loss_history = mlp_loss_histories[best_fold_idx]

plt.figure(figsize=(10, 6))
plt.plot(best_loss_history['train'], label='Training Loss', linewidth=2)
plt.plot(best_loss_history['val'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Log Loss')
plt.title(f'MLP Training/Validation Loss Curves (Fold {best_fold_idx + 1})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save loss history to CSV
loss_history_df = pd.DataFrame({
    'epoch': range(1, len(best_loss_history['train']) + 1),
    'train_loss': best_loss_history['train'],
    'val_loss': best_loss_history['val'],
})

# Save to results
results_path = Path(PROJECT_PATH) / 'results'
results_path.mkdir(parents=True, exist_ok=True)
loss_history_df.to_csv(results_path / 'mlp_loss_history.csv', index=False)

print(f"\nLoss history saved to: {results_path / 'mlp_loss_history.csv'}")
print(f"Final training loss: {best_loss_history['train'][-1]:.4f}")
print(f"Final validation loss: {best_loss_history['val'][-1]:.4f}")

## 6. Model Comparison

In [ ]:
# Compare LightGBM vs MLP
comparison_df = pd.DataFrame({
    'Metric': ['Macro F1', 'Log Loss', 'Accuracy', 'Precision', 'Recall', 'Train Time (s)'],
    'LightGBM': [
        lgbm_cv_df['macro_f1'].mean(),
        lgbm_cv_df['log_loss'].mean(),
        lgbm_cv_df['accuracy'].mean(),
        lgbm_cv_df['precision'].mean(),
        lgbm_cv_df['recall'].mean(),
        lgbm_cv_df['train_time_s'].sum(),
    ],
    'MLP': [
        mlp_cv_df['macro_f1'].mean(),
        mlp_cv_df['log_loss'].mean(),
        mlp_cv_df['accuracy'].mean(),
        mlp_cv_df['precision'].mean(),
        mlp_cv_df['recall'].mean(),
        mlp_cv_df['train_time_s'].sum(),
    ]
})

print("\n" + "=" * 60)
print("Model Comparison (5-fold CV average):")
print("=" * 60)
print(comparison_df.to_string(index=False))

# Determine winner
if lgbm_cv_df['macro_f1'].mean() > mlp_cv_df['macro_f1'].mean():
    winner = 'LightGBM'
    winner_cv_score = lgbm_cv_df['macro_f1'].mean()
else:
    winner = 'MLP'
    winner_cv_score = mlp_cv_df['macro_f1'].mean()

print(f"\nWinner by CV Macro F1: {winner} ({winner_cv_score:.4f})")
print("\nNote: Final winner selection should be done jointly with Member C")
print("      considering all 5 models (LogReg, DTree, RF, LightGBM, MLP)")

## 7. Evaluation on Student Test Set

In [ ]:
# Evaluate on student test set
print("Evaluating on Student Test Set...")
print("=" * 60)

# Use best models from CV (averaged predictions)

# LightGBM predictions
lgbm_test_probas = []
for model in lgbm_models:
    proba = model.predict(X_student, num_iteration=model.best_iteration)
    lgbm_test_probas.append(proba)

lgbm_test_proba_avg = np.mean(lgbm_test_probas, axis=0)
lgbm_test_pred = (lgbm_test_proba_avg >= 0.5).astype(int)

# MLP predictions
mlp_test_probas = []
for model, scaler in zip(mlp_models, mlp_scalers):
    X_student_scaled = scaler.transform(X_student)
    proba = model.predict_proba(X_student_scaled)[:, 1]
    mlp_test_probas.append(proba)

mlp_test_proba_avg = np.mean(mlp_test_probas, axis=0)
mlp_test_pred = (mlp_test_proba_avg >= 0.5).astype(int)

# Metrics
lgbm_student_metrics = {
    'macro_f1': f1_score(y_student, lgbm_test_pred, average='macro'),
    'log_loss': log_loss(y_student, lgbm_test_proba_avg),
    'accuracy': accuracy_score(y_student, lgbm_test_pred),
    'top1_accuracy': accuracy_score(y_student, lgbm_test_pred),  # Same as accuracy for binary
}

mlp_student_metrics = {
    'macro_f1': f1_score(y_student, mlp_test_pred, average='macro'),
    'log_loss': log_loss(y_student, mlp_test_proba_avg),
    'accuracy': accuracy_score(y_student, mlp_test_pred),
    'top1_accuracy': accuracy_score(y_student, mlp_test_pred),
}

student_comparison_df = pd.DataFrame({
    'Metric': ['Macro F1', 'Log Loss', 'Accuracy', 'Top-1 Accuracy'],
    'LightGBM': [
        lgbm_student_metrics['macro_f1'],
        lgbm_student_metrics['log_loss'],
        lgbm_student_metrics['accuracy'],
        lgbm_student_metrics['top1_accuracy'],
    ],
    'MLP': [
        mlp_student_metrics['macro_f1'],
        mlp_student_metrics['log_loss'],
        mlp_student_metrics['accuracy'],
        mlp_student_metrics['top1_accuracy'],
    ]
})

print("\nStudent Test Set Results:")
print(student_comparison_df.to_string(index=False))

## 8. Bayesian Hyperparameter Optimization (Optuna)

### 8.1 Define Search Space

In [ ]:
# Optuna search space for LightGBM (or winner model)
# This search space is justified based on:
# 1. LightGBM documentation recommendations
# 2. Common ranges for gradient boosting
# 3. Dataset characteristics (tabular, ~1-3M samples)

def objective_lgbm(trial: optuna.Trial) -> float:
    """
    Optuna objective function for LightGBM.
    
    Returns:
        CV macro F1 score (to maximize)
    """
    # Hyperparameter search space
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': seed,
        'n_jobs': -1,
        'verbose': -1,
    }
    
    # Cross-validation
    fold_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        X_fold_train = X_train[train_idx]
        y_fold_train = y_train[train_idx]
        X_fold_val = X_train[val_idx]
        y_fold_val = y_train[val_idx]
        
        train_data = lgb.Dataset(X_fold_train, label=y_fold_train)
        val_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_data)
        
        model = lgb.train(
            params={k: v for k, v in params.items() if k not in ['n_estimators']},
            train_set=train_data,
            num_boost_round=params['n_estimators'],
            valid_sets=[val_data],
            valid_names=['valid'],
            callbacks=[
                lgb.early_stopping(stopping_rounds=10, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )
        
        y_val_pred_proba = model.predict(X_fold_val, num_iteration=model.best_iteration)
        y_val_pred = (y_val_pred_proba >= 0.5).astype(int)
        
        fold_f1 = f1_score(y_fold_val, y_val_pred, average='macro')
        fold_scores.append(fold_f1)
    
    return np.mean(fold_scores)

print("Search Space Defined:")
print("  num_leaves: [15, 127]")
print("  max_depth: [-1, 15]")
print("  learning_rate: [0.01, 0.3] (log scale)")
print("  n_estimators: [50, 300]")
print("  min_child_samples: [5, 50]")
print("  subsample: [0.6, 1.0]")
print("  colsample_bytree: [0.6, 1.0]")
print("  reg_alpha: [1e-8, 10.0] (log scale)")
print("  reg_lambda: [1e-8, 10.0] (log scale)")
print("\nObjective: CV Macro F1 (maximize)")
print("Sampler: TPE (Tree-structured Parzen Estimator)")

### 8.2 Run Optuna Study

In [ ]:
# Run Optuna study
n_trials = 40  # 30-50 trials as specified

print(f"Running Optuna optimization with {n_trials} trials...")
print("This may take a while. Progress will be shown below.")
print("=" * 60)

# Create study
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=seed),
    study_name='lgbm_optimization'
)

# Run optimization
study.optimize(
    objective_lgbm,
    n_trials=n_trials,
    show_progress_bar=True,
)

print("\n" + "=" * 60)
print("Optimization Complete!")
print("=" * 60)
print(f"Best trial: {study.best_trial.number}")
print(f"Best CV Macro F1: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

### 8.3 Save Optuna Study

In [ ]:
# Save study results
hpo_path = Path(PROJECT_PATH) / 'hpo' / 'studies'
hpo_path.mkdir(parents=True, exist_ok=True)

# Save study database
import pickle

with open(hpo_path / 'study.pkl', 'wb') as f:
    pickle.dump(study, f)

# Save trials as CSV
trials_df = study.trials_dataframe()
trials_df.to_csv(hpo_path / 'trials.csv', index=False)

print(f"Study saved to: {hpo_path}")
print(f"  - study.pkl (full study object)")
print(f"  - trials.csv (trial results)")

# Also save to results
results_path = Path(PROJECT_PATH) / 'results'
results_path.mkdir(parents=True, exist_ok=True)
trials_df.to_csv(results_path / 'optuna_trials.csv', index=False)

print(f"  - results/optuna_trials.csv (copy)")

### 8.4 Optimization History Plot

In [ ]:
# Plot optimization history
plt.figure(figsize=(12, 5))

# Optimization history
plt.subplot(1, 2, 1)
trials_df = study.trials_dataframe()
plt.scatter(trials_df['number'], trials_df['value'], alpha=0.6, label='Trials')
plt.plot(trials_df['number'], trials_df['value'].cummax(), 'r-', linewidth=2, label='Best')
plt.xlabel('Trial')
plt.ylabel('CV Macro F1')
plt.title('Optimization History')
plt.legend()
plt.grid(True, alpha=0.3)

# Parameter importance
plt.subplot(1, 2, 2)
try:
    importance = optuna.importance.get_param_importances(study)
    params = list(importance.keys())
    values = list(importance.values())
    plt.barh(params, values)
    plt.xlabel('Importance')
    plt.title('Hyperparameter Importance')
except Exception as e:
    plt.text(0.5, 0.5, f'Importance\ncalculation\nfailed:\n{e}', 
             ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Hyperparameter Importance')

plt.tight_layout()
plt.show()

## 9. Train Final Model with Best Hyperparameters

In [ ]:
# Compare tuned vs untuned
print("=" * 60)
print("Tuned vs Untuned Comparison:")
print("=" * 60)

# Untuned performance (from CV)
untuned_cv_f1 = lgbm_cv_df['macro_f1'].mean()
untuned_log_loss = lgbm_cv_df['log_loss'].mean()

# Tuned performance
tuned_cv_f1 = study.best_value

# Evaluate tuned model on student test set
best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'random_state': seed,
    'n_jobs': -1,
    'verbose': -1,
})

# Train final model on all training data
print("\nTraining final tuned model on all 80 training trips...")

train_data = lgb.Dataset(X_train, label=y_train)

final_model = lgb.train(
    params={k: v for k, v in best_params.items() if k not in ['n_estimators']},
    train_set=train_data,
    num_boost_round=best_params['n_estimators'],
)

# Evaluate on student test
y_student_pred_proba = final_model.predict(X_student)
y_student_pred = (y_student_pred_proba >= 0.5).astype(int)

tuned_student_metrics = {
    'macro_f1': f1_score(y_student, y_student_pred, average='macro'),
    'log_loss': log_loss(y_student, y_student_pred_proba),
    'top1_accuracy': accuracy_score(y_student, y_student_pred),
}

# Comparison table
comparison_table = pd.DataFrame({
    'Metric': ['CV Macro F1', 'Student Macro F1', 'Student Log Loss', 'Student Top-1'],
    'Untuned': [
        untuned_cv_f1,
        lgbm_student_metrics['macro_f1'],
        lgbm_student_metrics['log_loss'],
        lgbm_student_metrics['top1_accuracy'],
    ],
    'Tuned': [
        tuned_cv_f1,
        tuned_student_metrics['macro_f1'],
        tuned_student_metrics['log_loss'],
        tuned_student_metrics['top1_accuracy'],
    ],
})

print("\n" + comparison_table.to_string(index=False))

# Improvement
print(f"\nImprovement in CV Macro F1: {((tuned_cv_f1 - untuned_cv_f1) / untuned_cv_f1 * 100):.2f}%")

## 10. Save Artifacts

In [ ]:
# Save final model and artifacts
artifacts_path = Path(PROJECT_PATH) / 'artifacts'
artifacts_path.mkdir(parents=True, exist_ok=True)

# Save model
final_model.save_model(str(artifacts_path / 'final_model.txt'))

# Save hyperparameters
with open(artifacts_path / 'best_hyperparams.json', 'w') as f:
    json.dump(best_params, f, indent=2)

# Save feature names
with open(artifacts_path / 'feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

# Save metadata
metadata = {
    'model_type': 'LightGBM',
    'training_date': datetime.now().isoformat(),
    'cv_macro_f1': float(tuned_cv_f1),
    'student_macro_f1': float(tuned_student_metrics['macro_f1']),
    'student_log_loss': float(tuned_student_metrics['log_loss']),
    'student_top1': float(tuned_student_metrics['top1_accuracy']),
    'n_training_trips': 80,
    'features_version': 'v0',
    'git_commit': 'TODO: Get from git',
}

with open(artifacts_path / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nArtifacts saved to: {artifacts_path}")
print("  - final_model.txt (LightGBM model)")
print("  - best_hyperparams.json")
print("  - feature_names.json")
print("  - model_metadata.json")

## 11. Run Logger (Save to experiments/runs/)

In [ ]:
# Save run JSON to experiments/runs/
experiments_path = Path(PROJECT_PATH) / 'experiments' / 'runs'
experiments_path.mkdir(parents=True, exist_ok=True)

# Generate run ID
run_id = f"lgbm_tuned_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Run JSON schema (from experiments/README.md)
run_json = {
    'run_id': run_id,
    'model': 'LightGBM',
    'family': 'boosting',
    'author': 'Member D',
    'config_path': 'configs/models/lgbm.yaml',
    'seed': seed,
    'features_version': 'v0',
    'hyperparameters': best_params,
    'cv_macro_f1': float(tuned_cv_f1),
    'student_top1': float(tuned_student_metrics['top1_accuracy']),
    'student_macro_f1': float(tuned_student_metrics['macro_f1']),
    'log_loss': float(tuned_student_metrics['log_loss']),
    'train_time_s': float(lgbm_cv_df['train_time_s'].sum()),
    'infer_time_s': float(lgbm_cv_df['infer_time_s'].mean()),
    'git_commit': 'TODO: Get from git',
    'timestamp': datetime.now().isoformat(),
}

# Save JSON
with open(experiments_path / f'{run_id}.json', 'w') as f:
    json.dump(run_json, f, indent=2)

print(f"\nRun saved to: {experiments_path / f'{run_id}.json'}")
print("\nRun JSON:")
print(json.dumps(run_json, indent=2))

## 12. Summary and Next Steps

In [ ]:
# Final summary
print("=" * 70)
print("MEMBER D TASK COMPLETION SUMMARY")
print("=" * 70)

print("\n✓ COMPLETED TASKS:")
print("  1. LightGBM model implemented and trained")
print("  2. MLP model implemented and trained with loss curves")
print("  3. Cross-validation performed (5-fold GroupKFold)")
print("  4. Student test set evaluation completed")
print("  5. Bayesian optimization run (Optuna, TPE sampler)")
print(f"  6. Best hyperparameters found (CV F1: {tuned_cv_f1:.4f})")
print("  7. Artifacts saved to artifacts/")
print("  8. Run JSON saved to experiments/runs/")
print("  9. Loss history saved to results/mlp_loss_history.csv")
print("  10. Optuna trials saved to hpo/studies/")

print("\n✓ KEY METRICS:")
print(f"  - LightGBM CV Macro F1 (untuned): {untuned_cv_f1:.4f}")
print(f"  - LightGBM CV Macro F1 (tuned): {tuned_cv_f1:.4f}")
print(f"  - Improvement: {((tuned_cv_f1 - untuned_cv_f1) / untuned_cv_f1 * 100):.2f}%")
print(f"  - MLP CV Macro F1: {mlp_cv_df['macro_f1'].mean():.4f}")

print("\n✓ NEXT STEPS:")
print("  1. Coordinate with Member C for full model comparison")
print("  2. Participate in Gate G2 (winner selection)")
print("  3. If LightGBM wins: tuned model is ready")
print("  4. If another model wins: re-run Optuna on that model")
print("  5. Write report sections 3.2 (LightGBM, MLP) and 4 (Bayesian opt)")
print("  6. Create docs/feature_engineering.md")

print("\n" + "=" * 70)
print("Notebook execution complete!")
print("=" * 70)